<a href="https://colab.research.google.com/github/Samu24042/CienciaDeDatos/blob/main/icd_taller4_estimacion_distribuciones.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **¡¡ ANTES DE EMPEZAR !!**

Deshabilita autocompletar con IA. Esta acción te ayudará a aprender de verdad. Si estás en Colab sigue estos pasos:



1.   Ir a Herramientas \ Configuración \ Asistencia de IA
2.   Desactivar la casilla **"Mostrar autocompletado impulsado por IA"**
3.   Activar la casilla **"Ocultar funciones de IA generativa"**


# **MÓDULO 1: OTRAS DISTRIBUCIONES TÍPICAS**
La distribución Normal y la Uniforme no son las únicas distribuciones de probabilidad típicas de variables aleatorias. En Ciencia de Datos, nos encontraremos con fenómenos que tienen comportamientos muy distintos que se modelan con otras distribuciones.

Por ejemplo, veamos estas tres distribuciones fundamentales:

1. **Distribución Exponencial:** Se usa en ingeniería de confiabilidad para modelar el tiempo hasta que una máquina falla, o en teoría de colas para modelar el tiempo de espera en la fila de un banco. El parámetro que identifica a la distribución exponencial es $\lambda$, el promedio de eventos (fallas o personas que llegan a la fila) por unidad de tiempo.

2. **Distribución Beta:** La distribución Beta es extremadamente flexible, siendo capaz de modelar distintos fenómenos dependiendo del valor de sus parámetros $\alpha$ y $\beta$. Por ejemplo, cuando $\alpha$ y $\beta$ son menores a 1 se puede usar para modelar la tasa de conversión de una campaña política donde los usuarios tienden a polarizarse.

3. **Distribución Log-Normal:** Se usa, por ejemplo, para modelar la distribución de la riqueza (pocas personas tienen muchísimo dinero), el tamaño de los archivos descargados de internet, o los precios de las acciones a largo plazo. Al igual que la distribución normal, sus parámetros son la media $\mu$ y la desviación estándar $\sigma$.

Veamos cómo son las Funciones de Densidad de Probabilidad (PDFs) de estas distribuciones a continuación.

In [ ]:
# Importamos las librerías necesarias
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import expon, beta, lognorm

# Configuramos el tamaño de la figura
plt.figure(figsize=(15, 5))


# 1. GRÁFICA: DISTRIBUCIÓN EXPONENCIAL (lambda = 1)
# Parámetros:
l = 1 # lambda = 1
# Gráfica:
plt.subplot(1, 3, 1) # (1 fila, 3 columnas, gráfica 1)
x_expon = np.linspace(0, 5, 1000)
pdf_expon = expon.pdf(x_expon, scale=1/l) # scale = 1/lambda
plt.plot(x_expon, pdf_expon, 'b-', lw=2)
plt.fill_between(x_expon, pdf_expon, alpha=0.2, color='blue')
plt.title('Distribución Exponencial')
plt.ylabel('Densidad p(x)')
plt.xlabel('x')
plt.grid(True, alpha=0.3)


# 2. GRÁFICA: DISTRIBUCIÓN BETA (alpha = 0.5, beta = 0.5)
# Parámetros:
a = 0.5 # alpha = 0.5
b = 0.5 # beta = 0.5
# Gráfica
plt.subplot(1, 3, 2)
# Evitamos el 0 y 1 exactos porque la densidad tiende a infinito en los bordes para alpha<1, beta<1
x_beta = np.linspace(0.01, 0.99, 1000)
pdf_beta = beta.pdf(x_beta, a, b)
plt.plot(x_beta, pdf_beta, 'r-', lw=2)
plt.fill_between(x_beta, pdf_beta, alpha=0.2, color='red')
plt.title(r'Distribución Beta ($\alpha=0.5, \beta=0.5$)')
plt.xlabel('x')
plt.grid(True, alpha=0.3)


# 3. GRÁFICA: DISTRIBUCIÓN LOG-NORMAL (mu = 0.7 , sigma = 0.6)
# Parámetros:
mu = 0.7
sigma = 0.6
# Gráfica
plt.subplot(1, 3, 3)
x_lognorm = np.linspace(0, 5, 1000)
# En Python, la función log-normal tiene otros parámetros: s y scale.
# Estos parámetros se calculan a partir de mu y sigma así:
s_calculado =  np.sqrt(np.log(1 + (sigma**2 / mu**2)))
scale_calculado = mu / np.sqrt(1 + (sigma**2 / mu**2))
pdf_lognorm = lognorm.pdf(x_lognorm, s=s_calculado, scale=scale_calculado)
plt.plot(x_lognorm, pdf_lognorm, 'g-', lw=2)
plt.fill_between(x_lognorm, pdf_lognorm, alpha=0.2, color='green')
plt.title('Distribución Log-Normal')
plt.xlabel('x')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


# **MÓDULO 2: MOMENTOS DE UNA VARIABLE ALEATORIA**

Si bien las gráficas de las PDFs nos dan la idea exacta de cómo se comporta una variable aleatoria (qué valores son los más probables, dónde está sesgada, etc.), a menudo no podemos o no queremos estar graficando cada variable.

En Ciencia de Datos, es conveniente resumir la gráfica en **características numéricas** que nos den una idea de la forma de la distribución sin necesidad de verla. Estos descriptores se conocen como **Momentos**.

Los momentos son, literalmente, la huella digital de una variable aleatoria. Si conoces todos los momentos de una variable, conoces exactamente su distribución.

### Fórmulas de los Momentos
Si conocemos la función de densidad $f_X(x)$, el $n$-ésimo Momento *Central* (*central* se refiere a que el cálculo se realiza respecto a la media $\mu$), denotado como $m_n$, se calcula integrando:
$$ m_n = E[(X-\mu)^n] = \int_{-\infty}^{\infty} (x-\mu)^n \cdot f(x) dx $$

Pero si estamos trabajando empíricamente con **datos extraídos de una muestra** (tamaño $N$), estimamos los momentos así:
$$ m_n = \frac{1}{N} \sum_{i=1}^{N} (x_i-\mu)^n $$

### Los 4 Momentos Principales
1. **Media / Valor Esperado (1er Momento):** Indica el "centro de masa" o tendencia central de la distribución.
2. **Varianza (2do Momento *Central*):** Mide qué tan esparcidos o dispersos están los valores alrededor de la media.
3. **Sesgo / Asimetría (3er Momento *Central*):** Indica hacia dónde está "inclinada" o sesgada la distribución. (0 es simétrica, positivo es cola hacia la derecha, negativo cola hacia la izquierda).
4. **Curtosis (4to Momento *Central*):** Mide el "peso" o grosor de las colas de la distribución en comparación con una distribución normal. Valores positivos y altos de curtosis significan que los eventos extremos (outliers) son más probables de lo que dictaría una campana de Gauss. Valores negativos indican una distribución más plana y con colas ligeras.

Cabe resaltar que el **Sesgo** y la **Curtosis** se suelen *''estandarizar''* para que no dependan de las unidades de la variable aleatoria (por ejemplo, para que sea comparable el sesgo obtenido con una distribución de peso en kilos con los de una distribución de alturas en metros de forma directa). La estandarización usa las siguientes fórmulas:

*   **Sesgo Estandarizado**$\displaystyle\ =\frac{m_3}{\sigma^3}$
*   **Curtosis Estandarizada**$\displaystyle\ =\frac{m_4}{\sigma^4}-3$:





In [ ]:
# ==========================================
# --- INTUICIÓN VISUAL ---
# ==========================================

# ANTES de hacer cálculos, mira atentamente las 3 gráficas que generamos arriba.
# Usa tu intuición para responder las siguientes preguntas.
# Reemplaza el texto "[TU RESPUESTA AQUÍ]" con tu elección:
# Opciones válidas: 'Exponencial', 'Beta' o 'Log-Normal'

print("1. La distribución con mayor Media parece ser: [TU RESPUESTA AQUÍ]")
print("2. La distribución con mayor Varianza parece ser: [TU RESPUESTA AQUÍ]")
print("3. La distribución con mayor Asimetría (cola derecha más pesada) es: [TU RESPUESTA AQUÍ]")
print("4. La distribución con mayor Curtosis (eventos extremos atípicos) es: [TU RESPUESTA AQUÍ]")


In [ ]:
# ==========================================
# --- VERIFICACIÓN DE MOMENTOS CON PYTHON ---
# ==========================================
# Ahora, usemos Python para calcular los Momentos a partir de una
# simulación grande de datos. Así validaremos tus respuestas anteriores.

from scipy.stats import skew, kurtosis

N = 100000 # Tamaño de la muestra simulada

# 1. Generamos los datos aleatorios para cada distribución
datos_expon = expon.rvs(scale=1/l, size=N)
datos_beta = beta.rvs(a, b, size=N)
datos_lognorm = lognorm.rvs(s=s_calculado, scale=scale_calculado, size=N)

# Creamos un diccionario iterativo para calcular y mostrar todo en orden
distribuciones = {
    "Exponencial": datos_expon,
    "Beta": datos_beta,
    "Log-Normal": datos_lognorm
}

print("--- RESULTADOS DE LOS MOMENTOS EMPÍRICOS ---\n")

for nombre, datos in distribuciones.items():
    # Cálculo usando NumPy y SciPy
    m_media = np.mean(datos)
    m_varianza = np.var(datos)
    m_asimetria = skew(datos) # Calcula el sesgo estandarizado
    m_curtosis = kurtosis(datos) # Calcula la curtosis estandarizada

    skew

    print(f"Distribución {nombre}:")
    print(f"  - Media      (1er): {m_media:.4f}")
    print(f"  - Varianza   (2do): {m_varianza:.4f}")
    print(f"  - Sesgo  (3er): {m_asimetria:.4f}")
    print(f"  - Curtosis   (4to): {m_curtosis:.4f}\n")

# ¿Acertaste tus predicciones iniciales?


# **Módulo 3: Estimación de una Distribución a partir de Datos**

En Ciencia de Datos, el proceso suele ser inverso al que acabamos de hacer: **no conocemos la distribución teórica de antemano**. Lo único que tenemos es una tabla con datos crudos y nuestro trabajo es descubrir qué distribución los generó.

### Proceso de Ajuste (Fitting)
1. **Obtener los datos y graficarlos:** Hacer un histograma para ver la forma.
2. **Calcular los Momentos:** Extraer la media, varianza, sesgo y curtosis de la muestra.
3. **Seleccionar Distribuciones Candidatas:** Usando la información de la gráfica y de los momentos calculados (ej. si el sesgo es muy positivo, descartamos la Normal y pensamos en una Exponencial o Log-Normal).
4. **Ajustar (Fit) los Parámetros:** Encontrar los parámetros que hacen que la curva teórica se adapte lo mejor posible a nuestro histograma.
5. Evaluar qué tan bien resulta el ajuste. Inicialmente, una inspección visual es útil. Luego, puede usarse una prueba de bondad como la **Prueba Kolmogorov-Smirnov (K-S)**. Esta calcula la distancia máxima entre la curva teórica y los datos reales. Cuanto más cercano a $0$ sea el estadístico K-S, mejor es el ajuste.

Vamos a aplicar este proceso a un **Dataset Real**. Analizaremos el total de las cuentas (`total_bill`) pagadas en un restaurante.

In [ ]:
# Importamos seaborn que contiene datasets de prueba
import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import skew, kurtosis

# 1. CARGAMOS EL DATASET REAL
# Este dataset contiene información sobre cuentas de restaurante y propinas
df_restaurante = sns.load_dataset('tips')

# Extraemos la variable que nos interesa: Total de la cuenta (en dólares)
datos_cuentas = df_restaurante['total_bill'].dropna().values

print("Primeras 5 cuentas registradas:", datos_cuentas[:5])
print("Total de datos (N):", len(datos_cuentas))

# ==========================================
# PASO 1 y 2: EXPLORACIÓN Y MOMENTOS
# ==========================================

# Calculamos los 4 momentos a partir de los datos
media = np.mean(datos_cuentas)
var = np.var(datos_cuentas)
sesgo = skew(datos_cuentas)
curtosis = kurtosis(datos_cuentas)

print("\n--- MOMENTOS EMPÍRICOS DE LAS CUENTAS ---")
print(f"Media     : ${media:.2f}")
print(f"Varianza  : {var:.2f}")
print(f"Sesgo     : {sesgo:.4f}")
print(f"Curtosis  : {curtosis:.4f}")

# Graficamos el histograma para ver la forma empírica
plt.figure(figsize=(8, 5))
# density=True hace que el área del histograma sume 1 (lo convierte en una Función de Densidad - PDF)
plt.hist(datos_cuentas, bins=20, density=True, color='lightgray', edgecolor='black', alpha=0.7, label='Datos empíricos (Histograma)')
plt.title('Distribución Empírica de las Cuentas de Restaurante')
plt.xlabel('Total de la cuenta ($ USD)')
plt.ylabel('Densidad')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.show()

# --- PREGUNTA ---
# ¿Crees que una Distribución Normal clásica sería un buen modelo predictivo aquí? ¿por qué?


### Evaluación de Candidatos: Normal vs. Log-Normal

Al analizar los momentos, notamos un **sesgo positivo fuerte ($>1$)**. Esto nos indica una cola pesada hacia la derecha (dado, por ejemplo, a que algunos clientes piden botellas de vino caras que inflan la cuenta).

En Python, la librería `scipy.stats` tiene un método llamado `.fit()`. Este método toma los datos empíricos y, utilizando técnicas de optimización (como el *Método de Máxima Verosimilitud*), calcula automáticamente cuáles deberían ser los parámetros de la curva para ajustarse "lo mejor posible" al histograma empírico.

Vamos a enfrentar a dos modelos candidatos:
1. La Distribución Normal.
2. La Distribución Log-Normal.

*¿A qué modelo le apuestas?*

In [ ]:
from scipy.stats import norm, lognorm

# ==========================================
# PASO 3 y 4: AJUSTE (FIT) DE DISTRIBUCIONES
# ==========================================

# 1. Ajustamos la Distribución Normal
# .fit() devuelve los parámetros estimados: (media_estimada, desviacion_estimada)
parametros_normal = norm.fit(datos_cuentas)
media_norm, std_norm = parametros_normal

# 2. Ajustamos la Distribución Log-Normal
# .fit() devuelve: (forma 's', ubicacion 'loc', escala 'scale')
parametros_lognorm = lognorm.fit(datos_cuentas, floc=0) # Fijamos floc=0 porque las cuentas empiezan en $0
s_estimado, loc_estimado, scale_estimado = parametros_lognorm

# ==========================================
# PASO 5: VERIFICACIÓN VISUAL
# ==========================================

# Creamos un eje X continuo para trazar nuestras curvas teóricas (desde $0 hasta $60)
x_teorico = np.linspace(0, 60, 1000)

# Generamos los valores Y (las PDFs) usando los parámetros que acabamos de encontrar
pdf_normal = norm.pdf(x_teorico, loc=media_norm, scale=std_norm)
pdf_lognorm = lognorm.pdf(x_teorico, s=s_estimado, loc=loc_estimado, scale=scale_estimado)

# Graficamos todo
plt.figure(figsize=(10, 6))

# 1. El histograma real
plt.hist(datos_cuentas, bins=20, density=True, color='lightgray', edgecolor='black', alpha=0.5, label='Datos Reales (Histograma)')

# 2. La curva Normal
plt.plot(x_teorico, pdf_normal, 'r--', lw=2, label='Ajuste Normal')

# 3. La curva Log-Normal
plt.plot(x_teorico, pdf_lognorm, 'g-', lw=3, label='Ajuste Log-Normal')

plt.title('Estimación de Distribuciones: Normal vs Log-Normal', fontsize=14)
plt.xlabel('Total de la cuenta ($ USD)', fontsize=12)
plt.ylabel('Densidad de Probabilidad p(x)', fontsize=12)
plt.legend()
plt.grid(axis='both', alpha=0.3)

# Limitamos el eje X para que se vea igual que el histograma
plt.xlim(0, 60)
plt.show()

# ==========================================
# PASO 6: VERIFICACIÓN CUANTITATIVA (K-S TEST)
# ==========================================
from scipy.stats import kstest

print("\n--- PRUEBA DE BONDAD DE AJUSTE (KOLMOGOROV-SMIRNOV) ---")
# Probamos qué tan bien se ajusta la Normal
stat_norm, p_norm = kstest(datos_cuentas, 'norm', args=(media_norm, std_norm))
# Probamos qué tan bien se ajusta la Log-Normal
stat_lognorm, p_lognorm = kstest(datos_cuentas, 'lognorm', args=(s_estimado, loc_estimado, scale_estimado))

print(f"Modelo Normal     -> Distancia K-S: {stat_norm:.4f}")
print(f"Modelo Log-Normal -> Distancia K-S: {stat_lognorm:.4f}")

print("\n--- CONCLUSIÓN ---")
print("Visualmente notamos que la curva roja (Normal) abarca valores negativos imposibles y falla en la cola, subestimando la cantidad de cuentas caras.")
print("Además, la distancia K-S de la Log-Normal es MENOR que la de la Normal.")
print("Por lo tanto, la Log-Normal es matemáticamente un mejor modelo para estos datos.")

# ==========================================
# PREGUNTA: Cuál es la probabilidad de que la cuenta salga por entre $30 y $40 USD?
# - Aquí tu código que soluciona esta pregunta: -


# **Reto 1: Toma de Decisiones Financieras bajo Incertidumbre**

Hasta ahora hemos analizado variables aleatorias por separado. Sin embargo, en la vida real, los fenómenos interactúan entre sí.

### Contexto del Negocio
Una *Startup* de Inteligencia Artificial tiene un modelo de negocio que cobra una tarifa a los clientes por usar su API. A la vez, le cuesta dinero procesar esas peticiones en servidores de Amazon Web Services (AWS).

Se tienen dos variables aleatorias diarias:
1. **Los Ingresos Diarios ($I$):** Son muy estables y predecibles. Siguen una **Distribución Normal** con media $\mu = \$5.000$ y desviación estándar $\sigma = \$500$.
2. **Los Costos Diarios de AWS ($C$):** Dependen de picos de computación (por ejemplo, clientes subiendo archivos muy grandes). Siguen una **Distribución Log-Normal** fuertemente sesgada a la derecha. La mayoría de los días el costo es bajo, pero de vez en cuando hay picos muy altos. La media de la distribución es $\mu = \$3.830 $ y su desviación estándar es $\sigma = \$3.050 $.

El equipo financiero debe tomar una decisión entre estas opciones:

* **Opción A (Plan Variable):** Seguir pagando a AWS como se ha venido haciendo. Es decir, los costos se modelan a través de la **Log-Normal** descrita anteriormente.
* **Opción B (Plan Tarifa Plana):** AWS ofrece un seguro. Un pago fijo garantizado de **$\$4,200$** diarios. Tráfico ilimitado.

**Entregable:** Calcular mediante simulación (100000 días) la **Ganancia Diaria ($G = I - C$)** para ambas opciones y decidir cuál plan minimiza el riesgo de que la empresa pierda dinero en un día determinado ($G < 0$).

In [3]:
# --- CÓDIGO PROBLEMA AWS ---
import numpy as np

N = 100000
mu_I = 1000
sigma_I = 50
mu_log = 6
sigma_log = 1

C_B = 900
I = np.random.normal(mu_I, sigma_I, N)
C_A = np.random.lognormal(mu_log, sigma_log, N)

G_A = I - C_A
G_B = I - C_B

prob_A = np.mean(G_A < 0)
prob_B = np.mean(G_B < 0)

print("Prob pérdida Plan A:", prob_A)
print("Prob pérdida Plan B:", prob_B)

Prob pérdida Plan A: 0.18147
Prob pérdida Plan B: 0.02233


# **Reto 2: Decisiones Basadas en Datos**

Cuando tenemos suficientes datos reales, a menudo los usamos directamente para simular escenarios futuros. Esta técnica se llama **Bootstrapping** (muestreo con reemplazo).

### Contexto del Negocio

Un restaurante está considerando reformar su política salarial para los meseros. Sabemos que en un turno típico nocturno, **un mesero atiende exactamente a 20 mesas**.

Para este análisis, intervendrán **dos variables aleatorias** de nuestro dataset sobre las cuentas de un restaurante: el total de la cuenta (`total_bill`) y la propina dejada (`tip`).

* **Opción A (Modelo Tradicional):** El mesero gana *exclusivamente* las propinas dejadas por los clientes.

* **Opción B (Modelo "Servicio Incluido"):** Se prohíben las propinas. A cambio, el restaurante garantiza al mesero un **16% fijo** sobre el monto total de cada cuenta que atienda.

**Entregable:** Simular 10.000 turnos de trabajo usando los datos reales para responder: ¿Qué modelo tiene una mayor ganancia esperada? Y lo más importante para la estabilidad del empleado: ¿Cuál modelo tiene menor **riesgo** de que un mesero termine su turno con menos de $60 dólares en el bolsillo?

In [9]:
# --- CÓDIGO PROBLEMA RESTAURANTE ---
import numpy as np
import pandas as pd
import seaborn as sns

df = sns.load_dataset("tips")
N_SIM = 10000
N_TABLES = 20

ganancias_A = []
ganancias_B = []

for _ in range(N_SIM):
    sample = df.sample(N_TABLES, replace=True)

    G_A = sample["tip"].sum()
    G_B = (0.16 * sample["total_bill"]).sum()

    ganancias_A.append(G_A)
    ganancias_B.append(G_B)

ganancias_A = np.array(ganancias_A)
ganancias_B = np.array(ganancias_B)

print(f"Ganancia esperada A (propinas): {ganancias_A.mean():.2f}")
print(f"Ganancia esperada B (16% fijo): {ganancias_B.mean():.2f}")

print(f"Riesgo A (G < 60): {np.mean(ganancias_A < 60):.4f}")
print(f"Riesgo B (G < 60): {np.mean(ganancias_B < 60):.4f}")

Ganancia esperada A (propinas): 60.00
Ganancia esperada B (16% fijo): 63.34
Riesgo A (G < 60): 0.5183
Riesgo B (G < 60): 0.3168
